In [ ]:
pip install langchain langchain-openai faiss-cpu sentence-transformers pydantic tqdm python-dotenv

In [ ]:
import os
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime
from functools import lru_cache
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.memory import ConversationBufferMemory
from pydantic import BaseModel, Field
import logging
from collections import defaultdict
import time
import hashlib
from sentence_transformers import CrossEncoder  # For reranking

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class MonitoringMetrics:
    """Class to track and report system metrics"""
    def __init__(self):
        self.metrics = {
            'total_queries': 0,
            'cache_hits': 0,
            'retrieval_times': [],
            'generation_times': [],
            'error_count': 0,
            'query_lengths': []
        }
        self.error_log = []

    def log_query(self, query_length: int):
        self.metrics['total_queries'] += 1
        self.metrics['query_lengths'].append(query_length)

    def log_cache_hit(self):
        self.metrics['cache_hits'] += 1

    def log_retrieval_time(self, time_ms: float):
        self.metrics['retrieval_times'].append(time_ms)

    def log_generation_time(self, time_ms: float):
        self.metrics['generation_times'].append(time_ms)

    def log_error(self, error: Exception):
        self.metrics['error_count'] += 1
        self.error_log.append({
            'timestamp': datetime.now().isoformat(),
            'error': str(error)
        })

    def get_metrics(self) -> Dict[str, Any]:
        """Return computed metrics with aggregations"""
        metrics = self.metrics.copy()

        # Calculate averages
        metrics['avg_retrieval_time'] = (
            sum(metrics['retrieval_times']) / len(metrics['retrieval_times'])
            if metrics['retrieval_times'] else 0
        )
        metrics['avg_generation_time'] = (
            sum(metrics['generation_times']) / len(metrics['generation_times'])
            if metrics['generation_times'] else 0
        )
        metrics['avg_query_length'] = (
            sum(metrics['query_lengths']) / len(metrics['query_lengths'])
            if metrics['query_lengths'] else 0
        )
        metrics['cache_hit_rate'] = (
            metrics['cache_hits'] / metrics['total_queries']
            if metrics['total_queries'] > 0 else 0
        )

        return metrics

class ContextConfig(BaseModel):
    """Configuration for context engineering"""
    system_role: str = Field(default="You are a helpful AI assistant.")
    max_history: int = Field(default=5, description="Number of past messages to retain")
    retrieval_top_k: int = Field(default=5, description="Number of relevant documents to retrieve")
    rerank_top_k: int = Field(default=3, description="Number of documents to keep after reranking")
    temperature: float = Field(default=0.7, description="LLM temperature parameter")
    cache_size: int = Field(default=1000, description="Maximum number of queries to cache")
    cache_ttl: int = Field(default=3600, description="Cache time-to-live in seconds")

class ContextEngine:
    """Production-scale context engineering system with advanced features"""

    def __init__(
        self,
        config: ContextConfig,
        vector_store: Optional[FAISS] = None,
        embeddings: Optional[Embeddings] = None,
        cross_encoder_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    ):
        self.config = config
        self.llm = ChatOpenAI(
            model="gpt-4",
            temperature=config.temperature,
            api_key=os.getenv("OPENAI_API_KEY")
        )
        self.embeddings = embeddings or OpenAIEmbeddings()
        self.vector_store = vector_store
        self.memory = ConversationBufferMemory(
            return_messages=True,
            memory_key="chat_history",
            input_key="question",
            output_key="answer"
        )
        self.metrics = MonitoringMetrics()
        self.cross_encoder = CrossEncoder(cross_encoder_model)
        self._cache = LRUCache(maxsize=config.cache_size, ttl=config.cache_ttl)

    def add_documents(self, documents: List[Document]) -> None:
        """Add documents to the vector store for context retrieval"""
        if self.vector_store is None:
            self.vector_store = FAISS.from_documents(documents, self.embeddings)
        else:
            self.vector_store.add_documents(documents)

    def retrieve_context(self, query: str) -> List[Document]:
        """Retrieve and rerank relevant context from vector store"""
        start_time = time.time()

        if self.vector_store is None:
            return []

        # First-stage retrieval (vector similarity)
        retrieved_docs = self.vector_store.similarity_search(
            query,
            k=self.config.retrieval_top_k
        )

        # Rerank with cross-encoder
        if len(retrieved_docs) > 1:
            doc_texts = [doc.page_content for doc in retrieved_docs]
            pairs = [[query, doc_text] for doc_text in doc_texts]
            scores = self.cross_encoder.predict(pairs)

            # Combine scores with documents
            scored_docs = list(zip(retrieved_docs, scores))
            # Sort by score descending
            scored_docs.sort(key=lambda x: x[1], reverse=True)
            # Take top N after reranking
            retrieved_docs = [doc for doc, score in scored_docs[:self.config.rerank_top_k]]

        self.metrics.log_retrieval_time((time.time() - start_time) * 1000)
        return retrieved_docs

    def format_context(self, documents: List[Document]) -> str:
        """Format retrieved documents into context string"""
        context_str = "\n\n".join([doc.page_content for doc in documents])
        return f"Relevant context ({len(documents)} documents):\n{context_str}"

    def _generate_cache_key(self, query: str) -> str:
        """Generate a unique cache key considering query and recent history"""
        history = self.memory.load_memory_variables({})["chat_history"]
        history_str = "".join(str(msg.content) for msg in history[-self.config.max_history:])
        key_str = f"{query}::{history_str}"
        return hashlib.sha256(key_str.encode()).hexdigest()

    def generate_response(self, user_query: str) -> str:
        """Generate response with full context engineering pipeline"""
        self.metrics.log_query(len(user_query))
        cache_key = self._generate_cache_key(user_query)

        # Check cache first
        if cached_response := self._cache.get(cache_key):
            self.metrics.log_cache_hit()
            return cached_response

        try:
            start_time = time.time()

            # Step 1: Retrieve and rerank relevant context
            retrieved_docs = self.retrieve_context(user_query)
            formatted_context = self.format_context(retrieved_docs)

            # Step 2: Get conversation history
            history = self.memory.load_memory_variables({})["chat_history"]

            # Step 3: Build the prompt with context
            prompt = ChatPromptTemplate.from_messages([
                SystemMessage(content=self.config.system_role),
                *history[-self.config.max_history:],  # Last N messages
                SystemMessage(content=formatted_context),
                HumanMessage(content=user_query)
            ])

            # Step 4: Generate response
            chain = (
                {"question": RunnablePassthrough()}
                | prompt
                | self.llm
                | StrOutputParser()
            )
            response = chain.invoke(user_query)

            # Step 5: Update memory and cache
            self.memory.save_context(
                {"question": user_query},
                {"answer": response}
            )
            self._cache.set(cache_key, response)

            self.metrics.log_generation_time((time.time() - start_time) * 1000)
            return response

        except Exception as e:
            self.metrics.log_error(e)
            logger.error(f"Error generating response: {str(e)}")
            return "I encountered an error processing your request."

class LRUCache:
    """Simple LRU cache with TTL support"""
    def __init__(self, maxsize: int = 1000, ttl: int = 3600):
        self.maxsize = maxsize
        self.ttl = ttl
        self._cache = {}
        self._order = []
        self._timestamps = {}

    def get(self, key: str) -> Any:
        if key not in self._cache:
            return None

        # Check if expired
        if (datetime.now().timestamp() - self._timestamps[key]) > self.ttl:
            self._delete_key(key)
            return None

        # Update access order
        self._order.remove(key)
        self._order.append(key)
        return self._cache[key]

    def set(self, key: str, value: Any) -> None:
        if key in self._cache:
            self._order.remove(key)
        elif len(self._cache) >= self.maxsize:
            self._delete_key(self._order[0])

        self._cache[key] = value
        self._order.append(key)
        self._timestamps[key] = datetime.now().timestamp()

    def _delete_key(self, key: str) -> None:
        del self._cache[key]
        del self._timestamps[key]
        self._order.remove(key)

# Example usage with enhanced features
if __name__ == "__main__":
    # Configuration
    config = ContextConfig(
        system_role="You are an expert customer support assistant for ACME Corp.",
        max_history=3,
        retrieval_top_k=5,
        rerank_top_k=2,
        cache_size=100,
        cache_ttl=300  # 5 minutes
    )

    # Sample documents for context
    documents = [
        Document(page_content="ACME Corp refund policy: 30-day money back guarantee.", metadata={"source": "policy"}),
        Document(page_content="Shipping takes 3-5 business days for standard delivery.", metadata={"source": "shipping"}),
        Document(page_content="Customer support hours: 9AM-5PM EST Mon-Fri.", metadata={"source": "support"}),
        Document(page_content="Defective products may be returned for full refund or replacement.", metadata={"source": "returns"}),
        Document(page_content="Express shipping available for $9.99 (2-day delivery).", metadata={"source": "shipping"}),
    ]

    # Initialize context engine with advanced features
    engine = ContextEngine(config)
    engine.add_documents(documents)

    # Example conversation
    queries = [
        "What's your refund policy?",
        "How long does shipping take?",
        "I need help with a defective product",
        "What's your refund policy?",  # This should hit cache
        "Do you offer fast shipping options?"
    ]

    for query in queries:
        print(f"User: {query}")
        start_time = time.time()
        response = engine.generate_response(query)
        elapsed = (time.time() - start_time) * 1000
        print(f"Assistant: {response}")
        print(f"(Generated in {elapsed:.2f}ms)\n")

    # Display metrics
    print("\n=== System Metrics ===")
    metrics = engine.metrics.get_metrics()
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.2f}")
        else:
            print(f"{k}: {v}")